# Direct Standardization in Practice: Re-Ranking Two Countries' Mortality

*Demographic Analysis Series · Module 1 (Mortality) · Lesson 1.2b*
*Author: Jiajia Li | Published: September 19, 2026 | Reading time: about 25 minutes*

[Lesson 1.2a](/blog/demographic-m1l2a-direct-standardization) derived the direct standardization formula and showed that the choice of standard population can change levels and, when age-specific schedules cross, even rankings. This lesson puts the method to work on a real comparison: Malaysia and Australia in 1988.

By the end of this notebook you will be able to:

- Compute crude death rates from age-specific rates and population distributions
- Implement a validated `standardize_direct()` function in Python
- Reproduce the classic reversal: Malaysia has the lower crude rate but the higher standardized rate
- Test how sensitive the conclusion is to the choice of standard population


## 1. The Puzzle: A Younger Population Looks Healthier

In 1988, Malaysia's crude death rate was about 5 per 1,000 population; Australia's was about 7. Read at face value, mortality in Malaysia looks roughly a third lower. Yet Lesson 1.1c plotted the two age-specific mortality curves side by side: Malaysia's rates are higher at almost every age.

Both facts are true, and they contradict each other only if we forget what a crude rate averages over. The CDR multiplies each age-specific rate by that age group's share of the population, so a country with few old people reports few deaths regardless of how dangerous each age is. Australia in 1988 was much older than Malaysia.

We use the same data as Lesson 1.1c: age-specific death rates (Carmichael, Table 1.2) and population distributions (Table 1.3), by sex and nineteen age groups. Deaths are derived as rate times population, so all totals inherit the rounding of the printed tables.


In [ ]:
# ---- 1. Load Tables 1.2/1.3 and compute crude death rates ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
from IPython.display import display

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 11,
    'figure.dpi': 125,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'legend.frameon': False,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})
GREEN, RUST = '#35634e', '#a95b3d'

# Carmichael (2016), Ch. 1: Table 1.2 (rates per 1,000) and Table 1.3 (persons per 1,000,000), 1988.
rates_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,16.6,9.8,12.8,7.6
1-4,1.7,0.5,1.1,0.4
5-9,0.6,0.2,0.5,0.2
10-14,0.6,0.3,0.4,0.2
15-19,1.0,1.1,0.5,0.4
20-24,1.5,1.6,0.6,0.5
25-29,1.7,1.5,0.9,0.5
30-34,2.0,1.4,1.1,0.6
35-39,2.4,1.5,1.5,0.8
40-44,3.4,2.2,2.3,1.2
45-49,5.5,3.4,3.2,2.1
50-54,9.9,6.0,5.9,3.4
55-59,15.4,10.0,9.9,5.5
60-64,25.5,17.3,17.0,8.7
65-69,38.3,27.2,27.8,13.8
70-74,62.7,45.3,46.6,23.5
75-79,86.2,71.9,69.8,40.7
80-84,144.3,110.7,119.0,71.4
85+,173.4,186.6,142.8,147.7
"""
pop_csv = """
age,m_malaysia,m_australia,f_malaysia,f_australia
0,14254,7597,13464,7245
1-4,56773,30324,53804,28954
5-9,61207,37762,58012,35803
10-14,56002,38712,53613,36758
15-19,52776,43626,50772,41743
20-24,50193,40771,49060,39297
25-29,42748,42914,44839,41982
30-34,35704,40234,38326,39933
35-39,30427,38778,31269,38391
40-44,23858,36248,23075,34537
45-49,20473,27923,19767,26357
50-54,17052,23855,17190,22798
55-59,12615,22708,13604,21924
60-64,10043,21764,10513,22336
65-69,7373,17701,8498,19973
70-74,4608,12857,5246,16200
75-79,3430,8712,3976,12555
80-84,1399,4471,1675,7827
85+,963,2319,1399,6111
"""
rates = pd.read_csv(StringIO(rates_csv), index_col='age')
pop = pd.read_csv(StringIO(pop_csv), index_col='age')
deaths = pop * rates / 1000

exposure = pd.DataFrame({
    'Malaysia': pop['m_malaysia'] + pop['f_malaysia'],
    'Australia': pop['m_australia'] + pop['f_australia'],
})
structure = exposure / exposure.sum()
asdr = pd.DataFrame({
    'Malaysia': (deaths['m_malaysia'] + deaths['f_malaysia']) / exposure['Malaysia'] * 1000,
    'Australia': (deaths['m_australia'] + deaths['f_australia']) / exposure['Australia'] * 1000,
})

total_deaths = pd.Series({
    'Malaysia': deaths[['m_malaysia', 'f_malaysia']].sum().sum(),
    'Australia': deaths[['m_australia', 'f_australia']].sum().sum(),
})
cdr = total_deaths / exposure.sum() * 1000
ur = cdr['Malaysia'] / cdr['Australia']

summary = pd.DataFrame({
    'CDR / 1,000': cdr.round(2),
    'Share aged 65+ (%)': (structure.loc['65-69':].sum() * 100).round(1),
})
display(summary)
print(f"Unstandardized ratio (Malaysia / Australia) = {ur:.3f}")
assert np.allclose(cdr.to_numpy(), [4.952, 7.251], atol=1e-3)


In [ ]:
# Figure 1: the two age structures
fig, ax = plt.subplots(figsize=(8.6, 4.0), layout='constrained')
x = np.arange(len(structure))
ax.plot(x, structure['Malaysia'] * 100, color=GREEN, marker='o', markersize=4,
        linewidth=1.6, label='Malaysia')
ax.plot(x, structure['Australia'] * 100, color=RUST, marker='s', markersize=4,
        linewidth=1.6, linestyle='--', label='Australia')
ax.set_xticks(x)
ax.set_xticklabels(structure.index, rotation=45, ha='right')
ax.set_xlabel('Age group')
ax.set_ylabel('Share of total population (%)')
ax.legend(loc='upper left')
ax.set_title('Two countries, two very different age structures (1988)', loc='left', pad=14)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#e6e8e3', linewidth=0.7)
plt.show()


**Figure 1 | Age structures of Malaysia and Australia, 1988.** Each point is an age group's share of the country's total population, both sexes combined, computed from Carmichael Table 1.3. Malaysia's distribution peaks in childhood; Australia's is shifted toward older ages. The share aged 65+ is 3.9% in Malaysia and 10.9% in Australia, so old-age mortality weighs almost three times as heavily in Australia's crude rate.

This is the composition effect from Lesson 1.1b, now with real data: the crude CDR is a weighted average in which Australia hands much more weight to the ages where everyone dies.


## 2. Implementing `standardize_direct()`

The direct standardized rate for population $i$ under standard $s$ is:

$$M_s(i) = \sum_x m_i(x)\,p_s(x),$$

where $m_i(x)$ is $i$'s age-specific rate and $p_s(x)$ is the standard's share in age group $x$. The formula is one line of pandas; the function below earns its keep through validation. Weights must sit on the same labelled age index as the rates, must be nonnegative, and are normalized internally. Silent misalignment between a rate table and a weight table is the most common implementation bug in applied work, so the function refuses to guess.

Two algebraic facts double as unit tests. First, standardizing the standard population itself returns its own crude rate: $M_s(s) = M(s)$. Second, rescaling all weights by any constant (100, one million) leaves the result unchanged after normalization, which is why printed percentage tables can be used as printed.


In [ ]:
# ---- 2. A validated direct-standardization function ----
def standardize_direct(rates, weights):
    """Direct standardized rate: sum of age-specific rates times standard weights.

    rates: Series of age-specific rates, or DataFrame with one column per population.
    weights: Series of standard-population shares on the same age index.
    Rates and weights must use the same rate unit (e.g., per 1,000 population).
    """
    w = weights / weights.sum()
    if not rates.index.equals(w.index):
        raise ValueError('rates and weights must share an identical age index')
    if (w < 0).any():
        raise ValueError('standard weights must be nonnegative')
    if isinstance(rates, pd.DataFrame):
        # pitfall: DataFrame * Series aligns the series on the columns; mul(axis='index') aligns on ages.
        return rates.mul(w, axis='index').sum(axis=0)
    return (rates * w).sum()

# Unit checks on the Lesson 1.2a toy example: rates 1 and 20 with weights 0.8 / 0.2.
toy_rates = pd.Series([1.0, 20.0], index=pd.Index(['Younger', 'Older'], name='age'))
toy_w = pd.Series([80.0, 20.0], index=toy_rates.index)  # percentages, as printed
assert np.isclose(standardize_direct(toy_rates, toy_w), 4.8)
toy_df = pd.DataFrame({'A': [1.0, 20.0], 'B': [0.8, 16.0]}, index=toy_rates.index)
assert np.allclose(standardize_direct(toy_df, toy_w).to_numpy(), [4.8, 3.84])

# The application: both schedules on the Australian age structure.
on_aus = standardize_direct(asdr, structure['Australia'])
comparison = pd.DataFrame({'Crude CDR': cdr.round(2), 'On Australian structure': on_aus.round(2)})
display(comparison)
sr = on_aus['Malaysia'] / on_aus['Australia']
print(f"Standardized ratio (Malaysia / Australia) = {sr:.3f}")
assert np.isclose(on_aus['Australia'], cdr['Australia'])  # a population standardized to itself is its crude rate
assert np.isclose(sr, 1.459, atol=1e-3)


In [ ]:
# Figure 2: the reversal, crude vs standardized
labels = ['Crude rate', 'Standardized to\nAustralian structure']
vals_m = [cdr['Malaysia'], on_aus['Malaysia']]
vals_a = [cdr['Australia'], on_aus['Australia']]
x = np.arange(2)
width = 0.36
fig, ax = plt.subplots(figsize=(7.2, 4.0), layout='constrained')
b1 = ax.bar(x - width / 2, vals_m, width, color=GREEN, label='Malaysia')
b2 = ax.bar(x + width / 2, vals_a, width, color=RUST, label='Australia')
ax.bar_label(b1, fmt='%.2f', padding=3)
ax.bar_label(b2, fmt='%.2f', padding=3)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Deaths per 1,000 population')
ax.set_ylim(0, 12)
ax.legend(loc='upper left')
ax.set_title('Same countries, opposite rankings', loc='left', pad=14)
ax.set_axisbelow(True)
ax.grid(axis='y', color='#e6e8e3', linewidth=0.7)
plt.show()


**Figure 2 | The comparison reverses under a common age structure.** Left pair: crude death rates computed from Tables 1.2 and 1.3. Right pair: both mortality schedules applied to Australia's 1988 age structure. Malaysia moves from 32% below Australia (ratio 0.68) to 46% above it (ratio 1.46). The standardized Australian entry equals its crude rate by construction.

Carmichael's Chapter 2 computes the same reversal from vital-registration counts: crude rates of 4.84 and 7.16 per 1,000, and an Australia-standardized Malaysian rate of 10.42. Our values, derived from the rounded published tables, are 4.95, 7.25, and 10.58. The absolute levels differ slightly because the printed rates carry one decimal; the standardized ratios agree to two decimal places (1.46 either way, 1.52 with Malaysia as the standard). Reproducing a published number exactly requires the publisher's data precision, a point Lesson 1.2a's reporting checklist already flagged.

Read the pair of ratios against the pivotal value 1.0. The unstandardized ratio (0.68) and the standardized ratio (1.46) sit on opposite sides: age composition did not merely exaggerate a gap, it inverted the direction of the comparison.


## 3. Sensitivity: Four Standards, One Conclusion

A standardized level is conditional on its standard, so the honest question is not "what is Malaysia's standardized rate?" but "does the comparison survive a change of standard?" We rerun the calculation under four standards: Australia's own structure, Malaysia's own structure, the Segi world standard, and the WHO World Standard.

The two world standards are printed on five-year age groups, so deaths and exposure are first aggregated from 0 and 1–4 into 0–4 and the rates recomputed from the aggregated counts. Averaging the printed rates of 0 and 1–4 directly would weight the two groups equally rather than by exposure.

Lesson 1.2a proved that when one schedule is higher at every age, no choice of standard can reverse the ranking. The present schedules do cross: Australia is higher at 85+ and essentially tied through the young adult years. A reversal back is therefore possible in principle. The empirical question is whether any plausible standard produces one.


In [ ]:
# ---- 3. The comparison under four standard populations ----
# Segi and WHO weights as printed in WHO (2001), Table 1 (transcribed in Lesson 1.2a).
five_yr = [f'{x}-{x+4}' for x in range(0, 85, 5)] + ['85+']
segi = pd.Series([12, 10, 9, 9, 8, 8, 6, 6, 6, 6, 5, 4, 4, 3, 2, 1, 0.5, 0.5], index=five_yr)
who = pd.Series([8.86, 8.69, 8.60, 8.47, 8.22, 7.93, 7.61, 7.15, 6.59,
                 6.04, 5.37, 4.55, 3.72, 2.96, 2.21, 1.52, 0.91, 0.63], index=five_yr)

def to_five_yr(v):
    out = v.copy()
    out.loc['0-4'] = out.loc['0'] + out.loc['1-4']
    return out.loc[five_yr]

asdr5 = to_five_yr(deaths[['m_malaysia', 'f_malaysia', 'm_australia', 'f_australia']].assign(
    Malaysia=deaths['m_malaysia'] + deaths['f_malaysia'],
    Australia=deaths['m_australia'] + deaths['f_australia'])[['Malaysia', 'Australia']]) / to_five_yr(exposure) * 1000

plan = [
    ('Australia (own structure)', asdr, structure['Australia']),
    ('Malaysia (own structure)', asdr, structure['Malaysia']),
    ('Segi world standard', asdr5, segi),
    ('WHO world standard', asdr5, who),
]
sens = pd.DataFrame({name: standardize_direct(r, w) for name, r, w in plan}).T
sens['Ratio M / A'] = sens['Malaysia'] / sens['Australia']
display(sens.round(2))
assert sens['Ratio M / A'].between(1.45, 1.53).all()
assert np.isclose(sens.loc['Malaysia (own structure)', 'Malaysia'], cdr['Malaysia'])


In [ ]:
# Figure 3: levels by standard (left), the Malaysia/Australia ratio (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0), layout='constrained')
xs = np.arange(len(sens))
short = ['Australia', 'Malaysia', 'Segi', 'WHO']
width = 0.36
ax1.bar(xs - width / 2, sens['Malaysia'], width, color=GREEN, label='Malaysia')
ax1.bar(xs + width / 2, sens['Australia'], width, color=RUST, label='Australia')
ax1.set_xticks(xs)
ax1.set_xticklabels(short)
ax1.set_xlabel('Standard population')
ax1.set_ylabel('Deaths per 1,000 population')
ax1.legend(loc='upper right')
ax1.set_title('Standardized levels move with the standard', loc='left', pad=14)
ax1.set_axisbelow(True)
ax1.grid(axis='y', color='#e6e8e3', linewidth=0.7)

xs2 = np.arange(-1, len(sens))
ratios = [ur] + sens['Ratio M / A'].tolist()
ax2.plot(xs2, ratios, color='#303e36', marker='o', markersize=5, linewidth=1.6)
for xi, yi in zip(xs2, ratios):
    ax2.annotate(f'{yi:.2f}', xy=(xi, yi), xytext=(0, 7), textcoords='offset points',
                 ha='center', fontsize=9)
ax2.axhline(1.0, color='#7b8078', linewidth=1, linestyle=':')
ax2.set_xticks(xs2)
ax2.set_xticklabels(['Crude'] + short)
ax2.set_xlabel('Standard population')
ax2.set_ylabel('Ratio, Malaysia / Australia')
ax2.set_ylim(0, 2)
ax2.set_title('The ratio stays on one side of 1.0', loc='left', pad=14)
ax2.set_axisbelow(True)
ax2.grid(axis='y', color='#e6e8e3', linewidth=0.7)
plt.show()


**Figure 3 | Levels are standard-dependent; the comparative conclusion is not.** Left: standardized rates under four standards. The same Malaysian schedule yields 10.58 on Australia's structure, 7.50 on Segi, 8.70 on WHO, and 4.95 (its own crude rate) on Malaysia's structure, all per 1,000 population. Right: the Malaysia-to-Australia ratio. The crude ratio (0.68) sits below 1; every standardized ratio sits near 1.5 (1.46 to 1.52).

The levels answer a hypothetical: "how many deaths per 1,000 would this schedule produce in a population shaped like the standard?" The ratio answers the comparative question, and here it is stable because Malaysia's excess mortality is large at the young ages that all plausible standards weight heavily, while the ages where Australia leads carry little weight everywhere.

A defensible reporting line for an applied paper:

> We directly age-standardized the 1988 death rates of Malaysia and Australia to the WHO World Standard Population (Ahmad et al., 2001), recomputing rates on five-year age groups from aggregated deaths and exposure. Under the WHO standard the Malaysian rate exceeds the Australian by a factor of 1.47; under the Segi standard and each country's own age structure the factor ranges from 1.46 to 1.52, against a crude ratio of 0.68.

Two disciplines follow. Always name the standard next to any standardized level. And never compare levels across standards, even when, as here, the comparative conclusion happens to be robust.


## 4. Check Your Understanding

1. Why does Australia's standardized rate under the Australian standard equal its crude rate of 7.25 exactly?
2. Compute the standardized ratio using the pooled Malaysia + Australia population as the standard. Does it lie between the two own-structure ratios?
3. A colleague reports "Malaysia's standardized CDR is 10.6 per 1,000" with no mention of a standard. What exactly is wrong, and what two questions should you ask?

**Answers.** (1) Because $M_s(s) = \sum_x m_s(x)\,p_s(x)$ is precisely the weighted-average identity for $s$'s own crude rate: standardizing a population to itself is the identity operation. (2) It does here (1.48, from rates of 7.76 and 5.26), but no theorem forces an arbitrary standard's ratio between the two own-structure ratios once schedules cross, so check rather than assume. (3) A standardized level is undefined without its standard. Ask which standard population and which age groups were used, and whether the conclusion survives a different standard.


## Coming Next: When Age-Specific Rates Are Too Fragile

Direct standardization needs a full age-specific rate schedule for every population compared. In small populations or narrow subgroups, age-specific death counts are tiny and the schedule is mostly noise. Lesson 1.3a develops indirect standardization and the standardized mortality ratio (SMR): apply one reliable standard schedule to your population's structure and compare observed with expected deaths.

[Back to the Demographic Analysis Series](/blog/series/demographic) · [Review Lesson 1.2a](/blog/demographic-m1l2a-direct-standardization)

## References and Data Sources

1. Carmichael, G. A. (2016). *Fundamentals of Demographic Analysis: Concepts, Measures and Methods*. Springer. Data: Ch. 1, Table 1.2 (age-specific death rates, per 1,000) and Table 1.3 (population by age and sex, per 1,000,000), Malaysia and Australia, 1988. Worked example: Ch. 2, *Comparison: Standardization and Decomposition*, pp. 56–62. [Springer book and chapter listing](https://link.springer.com/book/10.1007/978-3-319-23255-3).
2. Wachter, K. W. (2014). *Essential Demographic Methods*. Harvard University Press. The course syllabus assigns §6.5 for age standardization; match the section heading to the edition you use. [Publisher's book page](https://www.hup.harvard.edu/books/9780674045576).
3. Ahmad, O. B., Boschi-Pinto, C., Lopez, A. D., Murray, C. J. L., Lozano, R., & Inoue, M. (2001). *Age Standardization of Rates: A New WHO Standard*. GPE Discussion Paper Series No. 31. World Health Organization. Standard weights: Table 1 (printed p. 10). [Original WHO PDF](https://cdn.who.int/media/docs/default-source/gho-documents/global-health-estimates/gpe_discussion_paper_series_paper31_2001_age_standardization_rates.pdf).

**Reproducibility.** All data are embedded in the code cells, so execution requires no external downloads. Dependencies are NumPy, pandas, Matplotlib, and Jupyter/IPython. Assertions check the crude rates (4.95 and 7.25 per 1,000), the self-standardization identity, the toy-example unit tests for `standardize_direct()`, and the stability of the standardized ratio (1.45 to 1.53) across all four standards.
